# Assignment 10 - Problem 2
## Running DeepSeek-R1 locally with Ollama

**Name:** Hussein Jawad

The task: install Ollama, download (pull) the DeepSeek-R1 model, run it and check with `ollama ps` that it is running, then use `ollama.chat()` from Python to show that the model gives an answer together with its chain of thought.

The question I ask the model:

> "Explain the difference between the General and Special theory of relativity."

Contents:
- Step 1 - Install Ollama
- Step 2 - Check my machine and choose the model size
- Step 3 - Pull the DeepSeek-R1 model
- Step 4 - Run the model and check it with `ollama ps`
- Step 5 - Ask the question with `ollama.chat()` and show the chain of thought
- Appendix - Concepts used in this problem

## Step 1 - Install Ollama

Ollama runs large language models on my own computer. I downloaded the installer from https://ollama.com/download and ran it. After the install, Ollama runs in the background and listens on `http://localhost:11434`. Both the `ollama` command and the Python library talk to it there.

I also install the Ollama Python library, which gives me `ollama.chat()`.

In [ ]:
# Install the Ollama Python library (for ollama.chat) and psutil (to read the machine's memory)
%pip install -q ollama psutil

The next cell only does something in Google Colab: there it installs Ollama with the official Linux script and starts the Ollama server. On my own computer, Ollama is already installed and running, so the cell is skipped.

In [ ]:
import shutil, subprocess, time   # system tools: find programs, run commands, wait

try:
    import google.colab           # this import only works inside Colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False              # running on my own computer

if IN_COLAB and not shutil.which("ollama"):                                                  # Colab, Ollama not installed yet
    subprocess.run("apt-get -qq install -y zstd pciutils > /dev/null", shell=True, check=True)   # tools the installer needs
    subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)       # official Linux install script
if IN_COLAB:
    subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)  # start the server in the background
    time.sleep(5)                                                                                 # give the server time to start

print("Running in Colab:", IN_COLAB)   # where this notebook runs

Check that Ollama is installed (the `!` runs a terminal command from the notebook):

In [ ]:
# Show the installed Ollama version
!ollama --version

## Step 2 - Check my machine and choose the model size

DeepSeek-R1 comes in several sizes (1.5b, 7b, 8b, 14b, 32b, 70b, 671b; b = billions of parameters). From the lab, the rule of thumb is that I need at least as much free memory (RAM, or GPU memory for speed) as the size of the download, plus some extra for the context. `deepseek-r1:8b` is a 5.2 GB download, so it needs about 8 GB of free memory or more.

In [ ]:
import platform, psutil   # operating system info and memory info

print("Operating system:", platform.system(), platform.release())                 # e.g. Windows 11
print("CPU cores       :", psutil.cpu_count(logical=True))                         # number of CPU threads
print("RAM total       :", round(psutil.virtual_memory().total / 1e9, 1), "GB")    # installed memory
print("RAM available   :", round(psutil.virtual_memory().available / 1e9, 1), "GB")  # memory free right now

if shutil.which("nvidia-smi"):                                                       # an NVIDIA GPU driver is installed
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True).stdout.strip()             # ask the driver for name and memory
    print("GPU             :", gpu)                                                 # e.g. NVIDIA L4, 23034 MiB
else:
    print("GPU             : no NVIDIA GPU found (the model will run on the CPU, which is slower)")

In [ ]:
MODEL = "deepseek-r1:8b"   # 8B is the size the assignment recommends; use "deepseek-r1:1.5b" if memory is low
print("Model:", MODEL)     # the model used in the rest of the notebook

## Step 3 - Pull the DeepSeek-R1 model

`ollama.pull()` downloads the model, the same as `ollama pull deepseek-r1:8b` in a terminal. The first time it downloads about 5.2 GB. If the model is already on the computer, it only checks it. I print each stage once to keep the output short.

In [ ]:
import ollama   # Ollama Python library: talks to the Ollama server on localhost:11434

last_status = None                                  # last stage printed
for progress in ollama.pull(MODEL, stream=True):    # download the model, receiving progress updates
    if progress.status != last_status:              # print each new stage only once
        print(progress.status)
        last_status = progress.status

`ollama list` shows the models that are now on my computer:

In [ ]:
# List the downloaded models (name, ID, size on disk)
!ollama list

## Step 4 - Run the model and check it with `ollama ps`

Sending an empty prompt loads the model into memory without asking it anything. `keep_alive="30m"` keeps it loaded for 30 minutes (the default is 5), so it is still running for the question in Step 5. Then `ollama ps` shows which models are running.

In [ ]:
start = time.time()                                             # start the timer
ollama.generate(model=MODEL, prompt="", keep_alive="30m")       # empty prompt = just load the model into memory
print(f"Model loaded in {time.time() - start:.1f} seconds")     # loading time

# Show the models that are running right now
!ollama ps

The same information from Python, with the share of the model that sits in GPU memory:

In [ ]:
for m in ollama.ps().models:                                              # every model that is loaded
    print("Model          :", m.model)                                     # name and tag
    print("Size in memory :", round(m.size / 1e9, 1), "GB")                # weights + context memory
    print("On the GPU     :", f"{m.size_vram / m.size:.0%}")               # 100% = fully on the GPU (fast)
    print("Context length :", m.context_length, "tokens")                  # how much text it can look at
    print("Unloads at     :", m.expires_at)                                # when keep_alive runs out

## Step 5 - Ask the question with `ollama.chat()` and show the chain of thought

`ollama.chat()` takes the model name and a list of messages. With `think=True`, DeepSeek-R1 returns its reasoning in a separate field:
- `response.message.thinking` - the chain of thought
- `response.message.content` - the final answer

The model first writes all of its thinking and then the answer, so this call can take a minute or more.

In [ ]:
QUESTION = "Explain the difference between the General and Special theory of relativity."   # my question

start = time.time()                                          # start the timer
response = ollama.chat(
    model=MODEL,                                             # deepseek-r1:8b
    messages=[{"role": "user", "content": QUESTION}],       # one user message
    think=True,                                              # return the chain of thought separately
    keep_alive="30m",                                        # keep the model loaded afterwards
)
seconds = time.time() - start                                # how long the whole call took

thinking = response.message.thinking or ""                   # chain of thought (empty if the model gave none)
answer = response.message.content                            # final answer

print(f"Time                : {seconds:.1f} s")                                        # wall-clock time
print(f"Prompt tokens       : {response.prompt_eval_count}")                           # tokens in my question
print(f"Generated tokens    : {response.eval_count}")                                  # thinking + answer tokens
print(f"Speed               : {response.eval_count / (response.eval_duration / 1e9):.1f} tokens/s")   # durations are in ns
print(f"Chain of thought    : {len(thinking):,} characters")                           # length of the thinking
print(f"Final answer        : {len(answer):,} characters")                             # length of the answer
print(f"Thinking kept apart : {'<think>' not in answer and len(thinking) > 0}")       # True = two separate fields

### The chain of thought

The thinking is long, so I print the beginning and the end of it.

In [ ]:
if len(thinking) > 2500:                                      # long: show the start and the end only
    print(thinking[:1800])                                    # first 1,800 characters
    print(f"\n[... {len(thinking) - 2500:,} characters skipped ...]\n")
    print(thinking[-700:])                                    # last 700 characters
else:
    print(thinking)                                           # short enough to show in full

### The final answer

In [ ]:
print(answer)   # the model's final answer, without the thinking

Finally, `ollama ps` again shows that the model is still loaded after answering:

In [ ]:
# Models running now (the model stays loaded until keep_alive runs out)
!ollama ps

## Appendix - Concepts used in this problem

### 1. Running a model locally

**Ollama.** A program that downloads and runs open LLMs on my own computer. It runs a small server on `http://localhost:11434`; the `ollama` command, `curl` and the Python library all send their requests there. Nothing leaves my computer, and there is no API key and no cost per request.

**Model name and tag.** `deepseek-r1:8b` = model `deepseek-r1`, tag `8b` (8 billion parameters). A name without a tag means `:latest`.

**Distilled models.** The full DeepSeek-R1 has 671B parameters, far too big for a laptop. The smaller versions (1.5b to 70b) are *distilled*: smaller open models trained to copy the reasoning of the big one.

**Quantization.** Storing each weight with fewer bits (Ollama models use about 4 bits by default). That is why an 8B model is only about a 5 GB download instead of about 16 GB at 16 bits per weight.

**Memory, GPU and CPU.** The whole model has to fit in memory. If it fits in GPU memory it runs fast; if not, part of it runs on the CPU, which is much slower.

### 2. The commands I used

| Command | What it does |
|---|---|
| `ollama --version` | shows that Ollama is installed |
| `ollama pull <model>` / `ollama.pull()` | downloads a model |
| `ollama list` | lists the downloaded models |
| `ollama ps` | lists the models loaded in memory right now |
| `ollama.generate(model, prompt="")` | loads a model without asking anything |
| `ollama.chat(model, messages, think=True)` | asks a question and returns thinking + answer |

How to read `ollama ps`:

| Column | Meaning |
|---|---|
| NAME | the model that is loaded |
| SIZE | memory used: the weights plus memory for the context, so more than the download size |
| PROCESSOR | `100% GPU` = all on the GPU (fast); a CPU share means part of it runs on the CPU (slower) |
| CONTEXT | how many tokens the model can look at |
| UNTIL | when Ollama will unload the model (`keep_alive`, 5 minutes by default) |

### 3. Reasoning and the chain of thought

**Reasoning model.** DeepSeek-R1 is trained to "think out loud" before it answers: it writes a chain of thought (working out the problem step by step, checking itself), and then the final answer. This often makes answers better on harder questions, but it costs many extra tokens, so it is slower.

**`think=True`.** Tells Ollama to return the thinking in `message.thinking` and only the answer in `message.content`. Older versions put the thinking inside `<think>...</think>` tags in the content.

**Response numbers.** `prompt_eval_count` = tokens in my question, `eval_count` = tokens the model generated (thinking + answer), `eval_duration` and `total_duration` are in nanoseconds. Speed = `eval_count / (eval_duration / 10^9)` tokens per second.

### 4. The question: Special vs General relativity

To judge whether the model's answer is right, the main points are:

| | Special relativity (1905) | General relativity (1915) |
|---|---|---|
| Covers | motion at constant velocity (inertial frames), no gravity | gravity and accelerated motion |
| Key ideas | the laws of physics are the same in all inertial frames; the speed of light is the same for every observer | gravity is the curvature of spacetime caused by mass and energy; equivalence principle (gravity and acceleration feel the same) |
| Results | time dilation, length contraction, E = mc² | light bending around the Sun, gravitational time dilation (GPS corrections), black holes, gravitational waves |

Special relativity is the special case of general relativity when gravity can be ignored.